# Notebook 17 – Mini Assessment
### 17_Mini_Assessment.ipynb

This notebook contains questions and coding exercises on Feature Engineering, with answers included below each one.

### Q1. What is Feature Engineering?

**Answer:** Feature Engineering is the process of creating, transforming, or combining raw data into input variables (features) that better represent the underlying problem to a machine learning model — improving its ability to learn useful patterns.

### Q2. Feature Engineering vs Feature Selection

**Answer:**
- **Feature Engineering** — *creating* new features from raw data (e.g., extracting `PurchaseMonth` from a date column).
- **Feature Selection** — *choosing* which existing features (raw or engineered) are actually useful, and discarding the rest.

In short: Feature Engineering grows the feature set; Feature Selection shrinks it back down to what matters.

### Q3. Why is domain knowledge important?

**Answer:** Domain knowledge helps identify which features are likely to be meaningful, which relationships make business sense, and which columns might cause leakage. A data scientist who understands retail, for instance, knows that "days since last purchase" is a strong churn signal — a purely automated approach might not surface that as clearly. Domain knowledge turns raw columns into genuinely useful engineered features.

### Q4. What is Feature Leakage?

**Answer:** Feature Leakage happens when a feature contains information that would not actually be available at prediction time — often because it indirectly encodes the target, or comes from data generated after the event you're trying to predict. It makes a model look much more accurate during testing than it will be in real-world use.

### Q5. What is an interaction feature?

**Answer:** An interaction feature captures the **combined effect** of two or more features, which isn't visible when looking at each one separately. For example, `Quantity * UnitPrice` (giving `TotalPrice`) is an interaction feature — the total value depends on *both* variables together, not either alone.

In [1]:
import pandas as pd
sample = pd.DataFrame({'Quantity': [2, 5, 1], 'UnitPrice': [10, 4, 20]})
sample['TotalPrice'] = sample['Quantity'] * sample['UnitPrice']  # interaction feature
sample

,Quantity,UnitPrice,TotalPrice
0,2,10,20
1,5,4,20
2,1,20,20


### Q6. What is binning?

**Answer:** Binning converts a continuous numeric variable into discrete categories (bins/buckets). It can simplify relationships for the model, reduce the effect of noise/outliers, and sometimes make non-linear relationships easier for simple models to capture.

In [2]:
sample = pd.DataFrame({'UnitPrice': [1.5, 4.0, 9.0, 25.0, 60.0]})
sample['PriceBin'] = pd.cut(sample['UnitPrice'], bins=[0, 5, 15, 100], labels=['Low', 'Medium', 'High'])
sample

,UnitPrice,PriceBin
0,1.5,Low
1,4.0,Low
2,9.0,Medium
3,25.0,High
4,60.0,High


### Q7. What is feature cardinality?

**Answer:** Cardinality refers to the number of unique values a categorical feature has. `Country` has **low cardinality** (a handful of unique values), while `StockCode` or `CustomerID` have **high cardinality** (thousands of unique values). High-cardinality categorical features can be tricky to encode directly (e.g., one-hot encoding would create thousands of columns) and often need techniques like frequency or target encoding instead.

### Q8. Explain target encoding.

**Answer:** Target encoding replaces each category with the **average value of the target** for that category (computed from the training set). For example, encoding `Country` by the average `TotalPrice` of customers from that country.

⚠️ It must be computed **only on training data** (often with cross-validation folds) to avoid leakage — otherwise the encoding itself leaks target information into the features.

In [4]:
sample = pd.DataFrame({
    'Country': ['UK', 'UK', 'France', 'France', 'Germany'],
    'TotalPrice': [100, 150, 40, 60, 80]
})
country_avg = sample.groupby('Country')['TotalPrice'].mean()
sample['Country_TargetEncoded'] = sample['Country'].map(country_avg)
sample

,Country,TotalPrice,Country_TargetEncoded
0,UK,100,125.0
1,UK,150,125.0
2,France,40,50.0
3,France,60,50.0
4,Germany,80,80.0


### Q9. Explain frequency encoding.

**Answer:** Frequency encoding replaces each category with **how often it appears** in the dataset (a count or proportion). Unlike target encoding, it doesn't use the target at all, so it carries no leakage risk — but it also carries less predictive signal since it only reflects popularity, not outcome.

In [5]:
sample = pd.DataFrame({'Country': ['UK', 'UK', 'France', 'France', 'Germany']})
freq = sample['Country'].value_counts()
sample['Country_FreqEncoded'] = sample['Country'].map(freq)
sample

,Country,Country_FreqEncoded
0,UK,2
1,UK,2
2,France,2
3,France,2
4,Germany,1


### Q10. When should log transformation be used?

**Answer:** Log transformation is used on **right-skewed numeric features** (where most values are small but a few are very large) — such as income, price, or total spend. Taking the log compresses large values and spreads out small ones, making the distribution closer to normal, which helps many models (especially linear ones) perform better and reduces the influence of extreme outliers.

In [6]:
import numpy as np
sample = pd.DataFrame({'TotalSpend': [10, 50, 200, 5000, 50000]})
sample['LogTotalSpend'] = np.log1p(sample['TotalSpend'])  # log1p handles zero values safely
sample

,TotalSpend,LogTotalSpend
0,10,2.397895
1,50,3.931826
2,200,5.303305
3,5000,8.517393
4,50000,10.819798


### Q11. What is the curse of dimensionality?

**Answer:** The curse of dimensionality refers to problems that arise as the number of features grows very large: data becomes sparse in high-dimensional space, distances between points become less meaningful, models need exponentially more data to generalize well, and overfitting becomes much more likely. It's a key reason feature selection and dimensionality reduction (like PCA) matter.

### Q12. Explain PCA.

**Answer:** **Principal Component Analysis (PCA)** is a dimensionality reduction technique. It transforms a set of (possibly correlated) features into a smaller set of new, uncorrelated variables called **principal components**, ordered by how much variance in the data they explain. The first few components usually capture most of the useful information, letting you reduce dimensionality while keeping most of the signal.

In [8]:
from sklearn.decomposition import PCA
import pandas as pd
sample = pd.DataFrame({
    'Feature1': [1, 2, 3, 4, 5],
    'Feature2': [2, 4, 6, 8, 10],   # correlated with Feature1
    'Feature3': [5, 3, 6, 2, 8]
})
pca = PCA(n_components=2)
components = pca.fit_transform(sample)
print("Explained variance ratio:", pca.explained_variance_ratio_)
components

Explained variance ratio: [0.74183628 0.25816372]


array([[-4.14262372,  1.69666399],
       [-2.71215297, -0.94033306],
       [ 0.40474175,  1.12968319],
       [ 1.16064291, -3.39011918],
       [ 5.28939203,  1.50410505]])

### Q13. What is feature importance?

**Answer:** Feature importance is a score indicating how much each feature contributes to a model's predictions. Tree-based models (like Random Forest) naturally produce this by measuring how much each feature reduces impurity (error) across all the splits that use it — features used in more, and more impactful, splits get a higher importance score.

In [9]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
X = pd.DataFrame({
    'Recency': [5, 40, 2, 90, 10],
    'Frequency': [10, 1, 8, 1, 6],
    'Monetary': [500, 20, 400, 15, 300]
})
y = [1, 0, 1, 0, 1]  
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)
pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

Recency      0.373626
Frequency    0.340659
Monetary     0.285714
dtype: float64

### Q14. Create five useful features for an e-commerce dataset.

**Exercise:** Given a transaction-level dataframe with `Quantity`, `UnitPrice`, `InvoiceDate`, and `CustomerID`, create five useful features.

**Answer:**

In [10]:
import pandas as pd
sample = pd.DataFrame({
    'CustomerID': [1, 1, 2, 2, 3],
    'Quantity': [2, 1, 5, 3, 10],
    'UnitPrice': [10.0, 25.0, 4.0, 6.0, 2.0],
    'InvoiceDate': pd.to_datetime(['2024-01-05', '2024-01-20', '2024-02-01', '2024-02-15', '2024-03-01'])
})
sample['TotalPrice'] = sample['Quantity'] * sample['UnitPrice']       # 1. Interaction feature: transaction value
sample['PurchaseMonth'] = sample['InvoiceDate'].dt.month              # 2. Extracted date feature: seasonality signal
sample['PurchaseDayOfWeek'] = sample['InvoiceDate'].dt.dayofweek      # 3. Extracted date feature: shopping day pattern
sample['IsWeekend'] = sample['PurchaseDayOfWeek'] >= 5                # 4. Derived binary flag from date
sample['PricePerUnit'] = sample['TotalPrice'] / sample['Quantity']    # 5. Ratio feature: average price paid per unit
sample

,CustomerID,Quantity,UnitPrice,InvoiceDate,TotalPrice,PurchaseMonth,PurchaseDayOfWeek,IsWeekend,PricePerUnit
0,1,2,10.0,2024-01-05,20.0,1,4,False,10.0
1,1,1,25.0,2024-01-20,25.0,1,5,True,25.0
2,2,5,4.0,2024-02-01,20.0,2,3,False,4.0
3,2,3,6.0,2024-02-15,18.0,2,3,False,6.0
4,3,10,2.0,2024-03-01,20.0,3,4,False,2.0


### Q15. Create customer-level aggregation features.

**Exercise:** Aggregate the transaction-level data above up to one row per customer.

**Answer:**

In [11]:
customer_agg = sample.groupby('CustomerID').agg(
    TotalSpend=('TotalPrice', 'sum'),
    AvgOrderValue=('TotalPrice', 'mean'),
    PurchaseCount=('InvoiceDate', 'count'),
    FirstPurchase=('InvoiceDate', 'min'),
    LastPurchase=('InvoiceDate', 'max')
).reset_index()
customer_agg

,CustomerID,TotalSpend,AvgOrderValue,PurchaseCount,FirstPurchase,LastPurchase
0,1,45.0,22.5,2,2024-01-05,2024-01-20
1,2,38.0,19.0,2,2024-02-01,2024-02-15
2,3,20.0,20.0,1,2024-03-01,2024-03-01


### Q16. Extract features from a date column.

**Exercise:** From `InvoiceDate`, extract useful time-based features.

**Answer:**

In [12]:
sample['Year'] = sample['InvoiceDate'].dt.year
sample['Month'] = sample['InvoiceDate'].dt.month
sample['DayOfWeek'] = sample['InvoiceDate'].dt.dayofweek
sample['Quarter'] = sample['InvoiceDate'].dt.quarter
sample['IsWeekend'] = sample['InvoiceDate'].dt.dayofweek >= 5
sample[['InvoiceDate', 'Year', 'Month', 'DayOfWeek', 'Quarter', 'IsWeekend']]

,InvoiceDate,Year,Month,DayOfWeek,Quarter,IsWeekend
0,2024-01-05,2024,1,4,1,False
1,2024-01-20,2024,1,5,1,True
2,2024-02-01,2024,2,3,1,False
3,2024-02-15,2024,2,3,1,False
4,2024-03-01,2024,3,4,1,False


### Q17. Create interaction features.

**Exercise:** Create at least two interaction features from `Quantity`, `UnitPrice`, and `PurchaseCount` (from the aggregated table).

**Answer:**

In [13]:
combined = sample.merge(customer_agg[['CustomerID', 'PurchaseCount']], on='CustomerID')
combined['Quantity_x_UnitPrice'] = combined['Quantity'] * combined['UnitPrice']    
combined['Quantity_x_PurchaseCount'] = combined['Quantity'] * combined['PurchaseCount'] 
combined[['CustomerID', 'Quantity', 'UnitPrice', 'PurchaseCount', 'Quantity_x_UnitPrice', 'Quantity_x_PurchaseCount']]

,CustomerID,Quantity,UnitPrice,PurchaseCount,Quantity_x_UnitPrice,Quantity_x_PurchaseCount
0,1,2,10.0,2,20.0,4
1,1,1,25.0,2,25.0,2
2,2,5,4.0,2,20.0,10
3,2,3,6.0,2,18.0,6
4,3,10,2.0,1,20.0,10


### Q18. Perform feature selection using multiple techniques.

**Exercise:** Using the customer-level table, apply at least two feature selection techniques.

**Answer:**

In [14]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
fs_df = customer_agg.copy()
fs_df['FutureSpend'] = [600, 300, 150] 
X = fs_df[['TotalSpend', 'AvgOrderValue', 'PurchaseCount']]
y = fs_df['FutureSpend']
print("Correlation with target:")
print(pd.concat([X, y], axis=1).corr(numeric_only=True)['FutureSpend'])
rf = RandomForestRegressor(random_state=42)
rf.fit(X, y)
print("\nFeature importance:")
print(pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False))

Correlation with target:
TotalSpend       0.905221
AvgOrderValue    0.817057
PurchaseCount    0.755929
FutureSpend      1.000000
Name: FutureSpend, dtype: float64

Feature importance:
TotalSpend       0.426050
AvgOrderValue    0.425210
PurchaseCount    0.148739
dtype: float64
